This notebook processes FITS images to:

	•	Subtract background and detect astronomical sources
	•	Convert pixel coordinates to RA/Dec using WCS
	•	Save source catalogs and brightest object lists
	•	Visualize extracted sources and background
	•	Analyze astrometric differences (ΔRA, ΔDEC) across multiple frames

Useful for assessing image quality and preparing input for cross-matching or variability studies.

# Libraries

In [ ]:
# File system utilities
import os  # File system operations
import time  # Timing and benchmarking
from itertools import cycle  # Cycling through iterables

# Numerical and data processing
import numpy as np  # Numerical arrays and computations
import pandas as pd  # Tabular data structures (DataFrames)
from scipy.stats import norm  # Gaussian distribution tools
from scipy.optimize import curve_fit  # Curve fitting
from scipy.ndimage import gaussian_filter  # Image smoothing

# Visualization
import matplotlib.pyplot as plt  # Plotting
from matplotlib.patches import Circle  # Drawing circles on images
from mpl_toolkits.axes_grid1 import make_axes_locatable  # Flexible axis layout

# Astronomy-related tools
from astropy.io import fits  # Reading and writing FITS files
from astropy.wcs import WCS  # World Coordinate System (WCS) handling
from astropy.stats import sigma_clipped_stats  # Robust statistics with outlier rejection
from astropy.visualization import ZScaleInterval, MinMaxInterval  # Image scaling techniques
from astropy.table import Table  # Structured tables for astronomy
import astropy.units as u  # Handling physical units
from astropy.coordinates import SkyCoord  # Celestial coordinate transformations

# Source extraction and catalogs
import sep  # Source extraction from astronomical images
from astroquery.gaia import Gaia  # Querying Gaia catalog

# Parallel computing and progress display
from concurrent.futures import ThreadPoolExecutor  # Multithreading for performance
from tqdm import tqdm  # Progress bar for loops

# Functions

In [ ]:
def subtract_background(fits_file, output_directory, bw, bh, fw, fh, data=None, header=None):
    """
    Performs background subtraction from a FITS image and saves diagnostic plots.

    Parameters:
    ----------
    fits_file : str
        Path to the input FITS file.
    output_directory : str
        Path to the output directory for saving visualizations and results.
    bw : int
        Background mesh block width (in pixels).
    bh : int
        Background mesh block height (in pixels).
    fw : int
        Width of the filter window used to smooth the background map.
    fh : int
        Height of the filter window used to smooth the background map.
    data : ndarray, optional
        Pre-loaded image data. If not provided, will be read from file.
    header : astropy.io.fits.Header, optional
        FITS header. If not provided, will be read from file.

    Returns:
    -------
    data_sub : ndarray
        Background-subtracted image.
    bkg : sep.Background
        Background object estimated using SEP.
    header : astropy.io.fits.Header
        FITS header from the original file.
    """

    print(f"Processing file: {fits_file}")

    # Load image and header from FITS if not already provided
    if data is None:
        with fits.open(fits_file) as hdul:
            data = hdul[0].data
            header = hdul[0].header

    # Extract clean filename (without extension)
    file_name = fits_file.split("/")[-1].replace(".fits", "").replace(".fit", "")

    # Validate input
    if data is None or data.size == 0:
        print(f"Warning: No data in FITS file {fits_file}")
        return None, None, None

    # Ensure data is in float32 format for SEP
    data = data.astype(np.float32)

    # Estimate background using SEP
    bkg = sep.Background(data, bw=bw, bh=bh, fw=fw, fh=fh)

    # Subtract background from image
    data_sub = data - bkg

    # Compute sigma-clipped statistics
    mean_data, median_data, std_data = sigma_clipped_stats(data, sigma=3.0)
    mean_data_sub, median_data_sub, std_data_sub = sigma_clipped_stats(data_sub, sigma=3.0)
    mean_bkg, median_bkg, std_bkg = sigma_clipped_stats(bkg.back(), sigma=3.0)

    # --- Histogram Visualization ---
    fig, axs = plt.subplots(1, 3, figsize=(15, 3))

    # Histogram: Original Image
    axs[0].hist(data.ravel(), bins=256, histtype='step', color='black')
    axs[0].axvline(x=mean_data, color='r', linestyle='--', label=f'Mean: {mean_data:.1f}')
    axs[0].axvline(x=mean_data + std_data, color='g', linestyle='--', label=f'STD: {std_data:.1f}')
    axs[0].axvline(x=mean_data - std_data, color='g', linestyle='--')
    axs[0].set_xlim([mean_data - 5 * std_data, mean_data + 5 * std_data])
    axs[0].tick_params(axis='x', rotation=45)
    axs[0].set_title('Initial frame')
    axs[0].set_xlabel('Pixel Value')
    axs[0].set_ylabel('Frequency')
    axs[0].legend(loc="upper right")

    # Histogram: Background
    axs[1].hist(bkg.back().ravel(), bins=256, histtype='step', color='black')
    axs[1].axvline(x=mean_bkg, color='r', linestyle='--', label=f'Mean: {mean_bkg:.1f}')
    axs[1].axvline(x=mean_bkg + std_bkg, color='g', linestyle='--', label=f'STD: {std_bkg:.1f}')
    axs[1].axvline(x=mean_bkg - std_bkg, color='g', linestyle='--')
    axs[1].set_xlim([mean_bkg - 5 * std_data, mean_bkg + 5 * std_data])
    axs[1].tick_params(axis='x', rotation=45)
    axs[1].set_title('Background')
    axs[1].set_xlabel('Pixel Value')
    axs[1].set_ylabel('Frequency')
    axs[1].legend(loc="upper right")

    # Histogram: After Subtraction
    axs[2].hist(data_sub.ravel(), bins=256, histtype='step', color='black')
    axs[2].axvline(x=mean_data_sub, color='r', linestyle='--', label=f'Mean: {mean_data_sub:.1f}')
    axs[2].axvline(x=mean_data_sub + std_data_sub, color='g', linestyle='--', label=f'STD: {std_data_sub:.1f}')
    axs[2].axvline(x=mean_data_sub - std_data_sub, color='g', linestyle='--')
    axs[2].set_xlim([mean_data_sub - 5 * std_data_sub, mean_data_sub + 5 * std_data_sub])
    axs[2].tick_params(axis='x', rotation=45)
    axs[2].set_title('Distribution after subtraction')
    axs[2].set_xlabel('Pixel Value')
    axs[2].set_ylabel('Frequency')
    axs[2].legend(loc="upper right")

    plt.tight_layout()
    plt.savefig(os.path.join(output_directory, f'{file_name}_pix_distrib.png'), dpi=300)
    plt.show()

    # --- Image Comparison (with colorbars) ---
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # Original image
    img0 = axs[0].imshow(data, cmap='gray', origin='lower')
    axs[0].set_title('Initial data')
    divider0 = make_axes_locatable(axs[0])
    cax0 = divider0.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(img0, cax=cax0)

    # Background-subtracted image
    img1 = axs[1].imshow(data_sub, cmap='gray', origin='lower')
    axs[1].set_title('Data after Background Subtraction')
    divider1 = make_axes_locatable(axs[1])
    cax1 = divider1.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(img1, cax=cax1)

    # Estimated background
    img2 = axs[2].imshow(bkg.back(), cmap='gray', origin='lower')
    axs[2].set_title('Background')
    divider2 = make_axes_locatable(axs[2])
    cax2 = divider2.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(img2, cax=cax2)

    plt.tight_layout()
    plt.savefig(os.path.join(output_directory, f'{file_name}_sub_comparison.png'), dpi=300)
    plt.show()

    return data_sub, bkg, header

def evaluate_background_params(data, bw, bh, fw, fh):
    """
    Evaluate background estimation parameters on a given image.

    This function applies SEP's background estimation with specified block
    size and filter parameters and returns the global RMS of the background model.
    This RMS value reflects the noise level in the estimated background
    and can be used to compare different parameter combinations.

    Parameters:
    ----------
    data : ndarray
        2D array of the image data.
    bw : int
        Width of the background estimation mesh block (in pixels).
    bh : int
        Height of the background estimation mesh block (in pixels).
    fw : int
        Width of the filter used to smooth the background map.
    fh : int
        Height of the filter used to smooth the background map.

    Returns:
    -------
    rms : float
        Global root-mean-square of the background (i.e., estimated noise level).
    params : tuple
        The parameter combination used: (bw, bh, fw, fh).
    """
    bkg = sep.Background(data, bw=bw, bh=bh, fw=fw, fh=fh)
    return bkg.globalrms, (bw, bh, fw, fh)


def find_best_background_params(fits_file, output_directory, data=None):
    """
    Finds the optimal background subtraction parameters by minimizing global RMS noise
    and generates a diagnostic plot showing how background noise varies with parameters.

    Parameters:
    ----------
    fits_file : str
        Path to the input FITS file.
    output_directory : str
        Directory to save output figures and results.
    data : ndarray, optional
        Image data array (if already loaded). If not provided, it will be read from the file.

    Returns:
    -------
    best_params : tuple
        Best parameter combination as (bw, bh, fw, fh).
    all_results : list
        List of tuples in the form (rms, params) for all tested combinations.
    """
    if data is None:
        with fits.open(fits_file) as hdul:
            data = hdul[0].data.astype(np.float32)

    file_name = fits_file.split("/")[-1].replace(".fits", "").replace(".fit", "")

    # Define candidate parameter values
    bw_values = [32, 64, 128]
    fw_values = [3, 5, 7]

    best_rms = np.inf
    best_params = None
    all_results = []

    # Evaluate background parameters in parallel
    with ThreadPoolExecutor() as executor:
        futures = []
        for bw in bw_values:
            for fw in fw_values:
                futures.append(executor.submit(evaluate_background_params, data, bw, bw, fw, fw))

        for future in futures:
            rms, params = future.result()
            all_results.append((rms, params))
            if rms < best_rms:
                best_rms = rms
                best_params = params

    # Avoid overly small mesh and filter sizes (fallback if needed)
    if best_params == (32, 32, 3, 3):
        best_params = (64, 64, 5, 5)
        best_rms = next(rms for rms, params in all_results if params == (64, 64, 5, 5))

    # Prepare data for visualization
    rms_values = [result[0] for result in all_results]
    param_values = [f'bw={result[1][0]}, fw={result[1][2]}' for result in all_results]
    best_index = rms_values.index(best_rms)

    # Plot RMS error vs parameter combinations
    plt.figure(figsize=(8, 4))
    plt.plot(param_values, rms_values, 'o-', label='All parameters')
    plt.plot(param_values[best_index], rms_values[best_index], 'r*', label='Best parameter', markersize=15)
    plt.xticks(rotation=45)
    plt.xlabel('Parameters (bw, fw)')
    plt.ylabel('Global RMS Error')
    plt.title('Error vs Parameters')
    plt.legend()
    plt.tight_layout()
    plt.grid(alpha=0.25, lw=1, ls="-")
    plt.savefig(os.path.join(output_directory, f'{file_name}_error_vs_params.png'))
    plt.show()

    return best_params, all_results

def save_global_differences(output_directory):
    """
    Save the global coordinate differences in RA and Dec to separate text files.

    This function assumes that global variables `ra_diff_global` and `dec_diff_global`
    are populated elsewhere in the workflow and writes them to disk as plain text.
    
    Parameters:
    ----------
    output_directory : str
        Directory where the output files will be saved.
    """
    global ra_diff_global, dec_diff_global

    np.savetxt(os.path.join(output_directory, "global_ra_diff.txt"), ra_diff_global, fmt='%f')
    np.savetxt(os.path.join(output_directory, "global_dec_diff.txt"), dec_diff_global, fmt='%f')


def process_directory(directory, output_directory, circle_radius=10, scale='zscale'):
    """
    Process all FITS files in the specified directory: perform background subtraction,
    extract sources, and compute coordinate bounds.

    For each FITS image:
    - The optimal background subtraction parameters are determined.
    - Background is subtracted using SEP.
    - Sources are extracted and saved to CSV.
    - RA/Dec bounds of the frame are printed for quick inspection.

    Parameters:
    ----------
    directory : str
        Path to the directory containing input FITS files.
    output_directory : str
        Directory where output files will be saved.
    circle_radius : int, optional
        Radius used for visualization (not used in this function directly).
    scale : str, optional
        Image scaling mode for future visualizations (e.g., 'zscale', 'minmax').
    """
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Get list of FITS files in the input directory
    fits_files = [f for f in os.listdir(directory) if f.endswith('.fit') or f.endswith('.fits')]
    total_files = len(fits_files)

    # Iterate through each FITS file and process it
    for i, filename in enumerate(tqdm(fits_files, desc="Processing files", total=total_files)):
        start_time = time.time()
        fits_file = os.path.join(directory, filename)

        # Read image data and header
        with fits.open(fits_file) as hdul:
            data = hdul[0].data.astype(np.float32)
            header = hdul[0].header

        # Placeholder for optional vignetting correction
        data_v = data  # e.g., data_v = remove_objects_and_correct_vignetting(...)

        # Find optimal background parameters and subtract background
        best_params, all_results = find_best_background_params(fits_file, output_directory, data=data_v)
        if best_params is not None:
            bw, bh, fw, fh = best_params
            data_sub, bkg, header = subtract_background(fits_file, output_directory, bw, bh, fw, fh, data=data_v, header=header)

            if data_sub is not None:
                # Extract and export objects
                objects_df = extract_and_export_objects(data_sub, bkg, header, output_directory, fits_file)

                # Determine RA/Dec boundaries for the current image
                ra_min, ra_max, dec_min, dec_max = find_ra_dec_bounds(header)

                print("RA max = ",  round(ra_max, 3))
                print("RA min = ",  round(ra_min, 3))
                print("DEC max = ", round(dec_max, 3))
                print("DEC min = ", round(dec_min, 3))

        end_time = time.time()
        print(f"Processed {filename} ({i+1}/{total_files}) in {end_time - start_time:.2f} seconds.")
        print("-------------------------------------------")

def plot_objects(image_data, objects, output_image, scale='zscale'):
    """
    Visualizes extracted objects on an astronomical image and saves the result to a file,
    with optional scaling mode.

    Parameters:
    ----------
    image_data : ndarray
        2D array containing the image pixel values.
    objects : list or DataFrame
        List of extracted sources (e.g., from SEP), each with x, y coordinates.
    output_image : str
        Full path to the output file where the figure will be saved.
    scale : str, optional
        Scaling method for image display. Options: 'zscale', 'minmax', or 'linear'.
    """
    from astropy.visualization import ZScaleInterval, MinMaxInterval

    # Select scaling method
    if scale == 'zscale':
        interval = ZScaleInterval()
    elif scale == 'minmax':
        interval = MinMaxInterval()
    else:
        interval = None

    if interval:
        vmin, vmax = interval.get_limits(image_data)
    else:
        vmin, vmax = 0, np.percentile(image_data, 99)

    # Plot the image
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image_data, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)

    # Overlay extracted object positions
    for obj in objects:
        if 'a' in obj and 'b' in obj:
            # Use ellipse parameters to define marker size
            radius = 3 * np.sqrt(obj['a'] * obj['b'])
        else:
            radius = 10  # Fallback radius

        e = Circle((obj['x'], obj['y']), radius=radius, edgecolor='green', facecolor='none', lw=0.8)
        ax.add_patch(e)

    ax.axis('off')
    plt.savefig(output_image, bbox_inches='tight', pad_inches=0, dpi=300)
    plt.close()


def extract_and_export_objects(data_sub, bkg, header, output_directory, fits_file):
    """
    Extracts sources from a background-subtracted image and saves their parameters to CSV files.
    Also creates a separate file for the brightest objects.

    Parameters:
    ----------
    data_sub : ndarray
        Background-subtracted image array.
    bkg : sep.Background
        SEP background model object.
    header : astropy.io.fits.Header
        FITS header containing WCS information.
    output_directory : str
        Directory where CSV files will be saved.
    fits_file : str
        Path to the input FITS file, used to generate output filenames.

    Returns:
    -------
    df : pandas.DataFrame
        DataFrame with all extracted source parameters.
    df_top10 : pandas.DataFrame
        DataFrame with the top brightest sources (~1%).
    """
    if data_sub is None or bkg is None or header is None:
        print("Invalid input data.")
        return None, None

    # Extraction configuration
    thresh = 5.0  # Detection threshold (in sigma)
    minarea = 20  # Minimum number of connected pixels
    err = bkg.globalrms  # RMS map
    deblend_nthresh = 32
    deblend_cont = 0.05
    clean = True
    clean_param = 1.0

    sep.set_extract_pixstack(1e7)  # Increase pixel stack limit for complex fields

    # Source extraction
    objects = sep.extract(data_sub, thresh, err=err, minarea=minarea,
                          deblend_nthresh=deblend_nthresh,
                          deblend_cont=deblend_cont,
                          clean=clean, clean_param=clean_param)

    if len(objects) == 0:
        print(f"No objects found in {fits_file}")
        return None, None
    else:
        print(f"{len(objects)} objects found in {fits_file}")

    # Convert pixel coordinates to RA/Dec
    wcs = WCS(header)
    objects_coords = wcs.all_pix2world(objects['x'], objects['y'], 1)

    # Photometry and magnitude calculation
    flux, flux_err, flag = sep.sum_circle(data_sub, objects['x'], objects['y'], 3.0, err=err, gain=1.0)
    mag = -2.5 * np.log10(flux)
    mag_err = 2.5 / np.log(10) * (flux_err / flux)

    # Construct DataFrame
    data = {
        'id': range(len(objects)),
        'x': objects['x'],
        'y': objects['y'],
        'ra': objects_coords[0],
        'dec': objects_coords[1],
        'flux': flux,
        'flux_err': flux_err,
        'mag': mag,
        'mag_err': mag_err,
        'flag': flag
    }

    # Optionally add shape parameters
    if 'a' in objects.dtype.names and 'b' in objects.dtype.names:
        data['a'] = objects['a']
        data['b'] = objects['b']

    df = pd.DataFrame(data)

    # Sort by brightness and extract top 1%
    df_sorted = df.sort_values(by='mag')
    top_amount = int(len(df_sorted) * 0.01)
    df_top10 = df_sorted.head(top_amount)

    # Prepare output filenames
    base_name = os.path.splitext(os.path.basename(fits_file))[0]
    output_csv_file = os.path.join(output_directory, f"{base_name}_objects.csv")
    output_top_csv_file = os.path.join(output_directory, f"{base_name}_top{top_amount}_objects.csv")

    # Save results
    df.to_csv(output_csv_file, index=False)
    df_top10.to_csv(output_top_csv_file, index=False)

    return df

def plot_matching_errors(matches_df, header, output_directory=".", file_base_name="unknown"):
    """
    Visualizes coordinate matching errors (differences in RA and Dec) using scatter plots.

    Parameters:
    ----------
    matches_df : pandas.DataFrame
        DataFrame containing columns 'ra_diff', 'dec_diff', 'x', and 'y'.
    header : astropy.io.fits.Header
        FITS header used to determine axis mapping for coordinate labeling.
    output_directory : str, optional
        Directory where the plot will be saved.
    file_base_name : str, optional
        Base name for the output file.
    """

    # Determine which axis corresponds to RA and Dec
    x_label = "RA Difference" if "RA" in header['CTYPE1'] else "Dec Difference"
    y_label = "RA Difference" if "RA" in header['CTYPE2'] else "Dec Difference"

    # Create plots for coordinate differences
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))

    axs[0].scatter(matches_df['ra_diff'] * 3600, matches_df['x'], s=1)
    axs[0].set_ylabel("X coordinate")
    axs[0].set_xlabel(f"{x_label} (arcsec)")
    axs[0].set_title("RA Difference vs X coordinate")
    axs[0].grid(True)

    axs[1].scatter(matches_df['dec_diff'] * 3600, matches_df['y'], s=1)
    axs[1].set_ylabel("Y coordinate")
    axs[1].set_xlabel(f"{y_label} (arcsec)")
    axs[1].set_title("Dec Difference vs Y coordinate")
    axs[1].grid(True)

    plt.tight_layout()
    output_path = os.path.join(output_directory, f"{file_base_name}_coords_err.png")
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0, dpi=300)
    plt.show()


def find_ra_dec_bounds(header):
    """
    Computes the RA and Dec bounds of an image based on its FITS header.

    Parameters:
    ----------
    header : astropy.io.fits.Header
        FITS header containing WCS information.

    Returns:
    -------
    ra_min, ra_max : float
        Minimum and maximum Right Ascension (RA) in degrees.
    dec_min, dec_max : float
        Minimum and maximum Declination (Dec) in degrees.
    """
    wcs = WCS(header)

    # Image dimensions
    naxis1 = header['NAXIS1']
    naxis2 = header['NAXIS2']

    # Define pixel coordinates for image corners
    corners = np.array([[1, 1], [naxis1, 1], [1, naxis2], [naxis1, naxis2]])

    # Convert pixel coordinates to celestial coordinates (RA, Dec)
    world_coords = wcs.all_pix2world(corners, 1)

    # Extract RA and Dec values
    ra_vals = world_coords[:, 0]
    dec_vals = world_coords[:, 1]

    # Compute coordinate bounds
    ra_min, ra_max = np.min(ra_vals), np.max(ra_vals)
    dec_min, dec_max = np.min(dec_vals), np.max(dec_vals)

    return ra_min, ra_max, dec_min, dec_max

In [ ]:
# Global plotting style settings for matplotlib
plt.rcParams.update({
    'figure.figsize': (8, 6),         # Default figure size (width, height) in inches
    'figure.dpi': 100,                # Resolution for on-screen figures
    'savefig.dpi': 300,               # Resolution for saved figures
    'font.size': 12,                  # Base font size
    'font.family': 'serif',           # Font family (e.g., serif, sans-serif)
    'axes.labelsize': 14,            # Font size for axis labels
    'axes.titlesize': 14,            # Font size for axis titles
    'xtick.labelsize': 12,           # Font size for x-axis tick labels
    'ytick.labelsize': 12,           # Font size for y-axis tick labels
    'legend.fontsize': 10,           # Font size for legend text
    'lines.linewidth': 1,            # Default line width
    'lines.markersize': 6,           # Default marker size
    'axes.grid': False,              # Disable grid by default
    'grid.alpha': 0.25,              # Grid line transparency
    'grid.linestyle': '--',          # Grid line style
    'grid.color': 'gray',            # Grid line color
    'axes.axisbelow': True,          # Draw grid lines below other elements
    'image.cmap': 'viridis',         # Default colormap for image display
    'errorbar.capsize': 3,           # Length of the caps on error bars
    'legend.loc': 'best',            # Automatic legend placement
})

# Implementation

In [ ]:
fits_directory = os.getcwd() + '/data/'
output_directory = os.getcwd() + '/output_scanner_error/'

In [ ]:
process_directory(fits_directory, output_directory,scale='zscale')

In [ ]:
# Path to directory containing FITS files
fits_dir = os.getcwd() + "output_scanner_error/top"

# Lists for storing extracted data from all FITS files
combined_delta_ra = []
combined_delta_dec = []
combined_x = []
combined_y = []
file_indices = []

# Iterate through all FITS files in the directory and extract data
for i, filename in enumerate(os.listdir(fits_dir)):
    if filename.endswith(".fits"):
        file_path = os.path.join(fits_dir, filename)

        with fits.open(file_path) as hdul:
            data = pd.DataFrame(hdul[1].data)

            # Convert RA/DEC differences to arcseconds
            data['delta_ra_arcsec'] = data['delta_ra'] * 3600
            data['delta_dec_arcsec'] = data['delta_dec'] * 3600

            # Aggregate RA/DEC offsets and coordinates
            combined_delta_ra.extend(data['delta_ra_arcsec'])
            combined_delta_dec.extend(data['delta_dec_arcsec'])

            # Determine correct x/y mapping
            if data['x_axis'].iloc[0] == 'ra':
                combined_x.extend(data['x'])
                combined_y.extend(data['y'])
            else:
                combined_x.extend(data['y'])
                combined_y.extend(data['x'])

            # Track the file index for color grouping (if needed)
            file_indices.extend([i] * len(data))

# Visualization with different colors and markers for each file
plt.figure(figsize=(14, 6))

# Define distinct colors and markers
colors = cycle([
    'red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink',
    'gray', 'olive', 'cyan', 'magenta', 'yellow', 'lime', 'navy',
    'teal', 'coral', 'gold', 'indigo', 'violet', 'turquoise'
])

markers = cycle([
    'o', 's', 'D', '^', 'v', '<', '>', 'P', '*', 'X', 'h', '+',
    'x', 'd', '|', '_', 'H', '8', 'p', '1'
])

# Plot ΔRA vs X
plt.subplot(1, 2, 1)
for i in range(len(combined_delta_ra)):
    plt.scatter(combined_delta_ra[i], combined_x[i],
                color=next(colors),
                marker=next(markers),
                alpha=0.7,
                label=f'File {file_indices[i] + 1}' if i == 0 else "")

plt.xlabel(r'$\Delta$RA (arcsec)')
plt.ylabel('Physical coordinate')
plt.title(r'Combined $\Delta$RA')

# Plot ΔDEC vs Y
plt.subplot(1, 2, 2)
for i in range(len(combined_delta_dec)):
    plt.scatter(combined_delta_dec[i], combined_y[i],
                color=next(colors),
                marker=next(markers),
                alpha=0.7,
                label=f'File {file_indices[i] + 1}' if i == 0 else "")

plt.xlabel(r'$\Delta$DEC (arcsec)')
plt.ylabel('Physical coordinate')
plt.title(r'Combined $\Delta$DEC')

plt.tight_layout()
plt.savefig(os.path.join(fits_dir, "20_data_combined.png"), dpi=300)
plt.show()

# Sigma-clipping statistics setup
sigma = 5

# Plot ΔRA and ΔDEC distributions with clipped statistics
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# ΔRA vs X coordinate
ax[0].scatter(combined_delta_ra, combined_x, color='steelblue', alpha=0.5)
ax[0].set_xlabel(r'$\Delta$RA (arcsec)')
ax[0].set_ylabel('Physical coordinate')
ax[0].set_title(r'All Data Combined: $\Delta$RA')

# Compute robust statistics for ΔRA
mean_ra, median_ra, std_ra = sigma_clipped_stats(combined_delta_ra, sigma=sigma)

# Annotate with vertical lines
ax[0].axvline(mean_ra, color='r', linestyle='--', label=f'Mean: {mean_ra:.3f}')
ax[0].axvline(median_ra, color='g', linestyle='-', label=f'Median: {median_ra:.3f}')
ax[0].axvline(mean_ra - std_ra, color='gray', linestyle='-.', lw=2, label=f'-{sigma}$\sigma$: {mean_ra - std_ra:.3f}')
ax[0].axvline(mean_ra + std_ra, color='gray', linestyle='-.', lw=2, label=f'+{sigma}$\sigma$: {mean_ra + std_ra:.3f}')
ax[0].legend()

# ΔDEC vs Y coordinate
ax[1].scatter(combined_delta_dec, combined_y, color='steelblue', alpha=0.5)
ax[1].set_xlabel(r'$\Delta$DEC (arcsec)')
ax[1].set_ylabel('Physical coordinate')
ax[1].set_title(r'All Data Combined: $\Delta$DEC')

# Compute robust statistics for ΔDEC
mean_dec, median_dec, std_dec = sigma_clipped_stats(combined_delta_dec, sigma=sigma)

# Annotate with vertical lines
ax[1].axvline(mean_dec, color='r', linestyle='--', label=f'Mean: {mean_dec:.3f}')
ax[1].axvline(median_dec, color='g', linestyle='-', label=f'Median: {median_dec:.3f}')
ax[1].axvline(mean_dec - std_dec, color='gray', linestyle='-.', lw=2, label=f'-{sigma}$\sigma$: {mean_dec - std_dec:.3f}')
ax[1].axvline(mean_dec + std_dec, color='gray', linestyle='-.', lw=2, label=f'+{sigma}$\sigma$: {mean_dec + std_dec:.3f}')
ax[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(fits_dir, "all_data_combined.png"), dpi=300)
plt.show()

# Print summary statistics to console
print("----- RA Statistics -----")
print(f"Maximum = {np.max(combined_delta_ra):.6f}")
print(f"Minimum = {np.min(combined_delta_ra):.6f}")
print(f"Mean = {np.mean(combined_delta_ra):.6f}")
print(f"Median = {np.median(combined_delta_ra):.6f}")
print(f"Mean - {sigma}*sigma = {mean_ra - std_ra:.6f}")
print(f"Mean + {sigma}*sigma = {mean_ra + std_ra:.6f}")

print("----- DEC Statistics -----")
print(f"Maximum = {np.max(combined_delta_dec):.6f}")
print(f"Minimum = {np.min(combined_delta_dec):.6f}")
print(f"Mean = {np.mean(combined_delta_dec):.6f}")
print(f"Median = {np.median(combined_delta_dec):.6f}")
print(f"Mean - {sigma}*sigma = {mean_dec - std_dec:.6f}")
print(f"Mean + {sigma}*sigma = {mean_dec + std_dec:.6f}")